In [22]:
import time
from pynq.overlays.base import BaseOverlay

base = BaseOverlay("base.bit")

the base overlay intiated.

In [28]:

%%microblaze base.PMODA

#include "gpio.h"
#include "pyprintf.h"

void write_gpio(unsigned int pin, unsigned int value){
    if (value > 1){
        pyprintf("pin vlaue must be 0 or 1/n");
        return;
    }
    gpio device = gpio_open(pin);
    
    gpio_set_direction(device, 0x0); //#<-- 0 is output
    gpio_write(device, value);
}

unsigned int read_gpio(unsigned int pin){
    
    gpio device = gpio_open(pin);
    
    gpio_set_direction(device, 0x1); //#<-- 1 is input
    return gpio_read(device);
}


In [29]:
import pynq
print(pynq.__version__)


3.1.1


In [30]:
buzzer = 3 #<-- buzzer on data pin 3

def play_tone(frequency_hz, duration_s):
    
    half_period = 1/ (2 * frequency_hz)
    stop_time = time.time() + duration_s
    
    while time.time() < stop_time:
        write_gpio(buzzer, 1)
        time.sleep(half_period)#<-- buzzer high
        
        write_gpio(buzzer, 0)
        time.sleep(half_period) #<-- buzzer low

quick sanity check below for some error messages i got early on.  checking pynq version on the board.

In [36]:
play_tone(900, 0.5)

now we move onto the socket server process below.

In [28]:
#btns = base.btns_gpio
#leds = base.leds
import socket

def server_process(local_addr= "127.0.0.1", port=12345):

    # PYNQ1_IP = "192.168.0.200"---------->PYNQ2_IP = "192.168.0.201"
    # PORT_1 = 12345---------------------->PORT_2 = 12345
    # server_process(PYNQ1_IP, PORT_1)---->server_process(PYNQ2_IP, PORT_2)
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((local_addr, port))
    server.listen(1)
    print(f"[SERVER] listening on {local_addr}:{port}")
    
    while True:
        connection, address = server.accept()
        print("Server connected by", local_addr)
        
        while True:
            data = connection.recv(1024)
            
            if not data:
                break
            
            message = data.decode(errors="ignore").strip()
            
            if message == "TONE":
                #play_tone(1000, 0.5)
                print("play tone here")
            elif message == "DISCONNECT":
                break
            
    connection.close()
    print("Client disconnected")

In [29]:
import socket, time
HOST = "127.0.0.1"  #<--server's IP
PORT = 12345

with socket.socket(socket.AF_INET, socket.SOCK_STREAM)as s:
    s.connect((HOST, PORT))
    print("[CLIENT] Connected")
    message =  "TONE"               #data.decode(errors="ignore").strip()
    s.sendall(message.encode())
    print ("Sent:", message.strip())
    
    time.sleep(0.2)
    message = "DISCONNECT"
    data = s.recieve(1024)
    s.sendall(message.encode())
    print ("Sent:",message)
   

ConnectionRefusedError: [Errno 111] Connection refused